# ADS 所有版本 XLSX 结构预检

只从已封存 ZIP 取唯一 XLSX 原件，核验成员大小/CRC；以只读方式查看至多 5 个工作表的前 10 行与字段角色，不读 SOXX 或拟合。

In [ ]:
from pathlib import Path
import hashlib,json,re,shutil,zipfile
import openpyxl
root=Path.cwd();card=root/'research/philadelphia_ads_vintage_source_20261023';plan_path=card/'vintage_schema_audit_plan.json';plan=json.loads(plan_path.read_text());parent=json.loads((card/plan['parent_source_result']).read_text());raw=root/'data/snapshots/philadelphia_ads_vintage_20261023/ADS_All_Vintages.zip'
assert plan['frozen'] and not (card/'schema_run_state.json').exists()
assert hashlib.sha256(raw.read_bytes()).hexdigest()==plan['raw_outer_ZIP_sha256']==parent['raw_ZIP_sha256'] and parent['scoped_raw_ZIP_restore_verified']
(card/'schema_run_state.json').write_text(json.dumps({'status':'nested_XLSX_extraction_started','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'outer_ZIP_reads':1,'nested_extraction_attempts':1,'workbook_open_attempts':1,'model_fits':0,'accounts':0},indent=2)+'\n')
nested=raw.parent/plan['nested_member'];assert not nested.exists()
with zipfile.ZipFile(raw) as z:
    info=z.getinfo(plan['nested_member']);assert info.file_size==plan['nested_uncompressed_bytes']
    with z.open(info) as source,nested.open('wb') as target:shutil.copyfileobj(source,target,1024*1024)
assert nested.stat().st_size==info.file_size
nested_sha=hashlib.sha256(nested.read_bytes()).hexdigest()
workbook=openpyxl.load_workbook(nested,read_only=True,data_only=True)
assert len(workbook.sheetnames)<=100
sheets=[]
for name in workbook.sheetnames[:plan['budget']['sheets_inspected_max']]:
    sheet=workbook[name];preview=[]
    for row in sheet.iter_rows(min_row=1,max_row=plan['budget']['rows_per_sheet_max'],values_only=True):
        preview.append([str(value)[:100] if value is not None else None for value in row[:24]])
    sheets.append({'name':name,'declared_max_row':sheet.max_row,'declared_max_column':sheet.max_column,'first_rows_first_24_columns':preview})
workbook.close()
flat=' '.join(str(x) for s in sheets for row in s['first_rows_first_24_columns'] for x in row if x is not None)
dates=re.findall(r'\b(?:19|20)\d{2}[-/]\d{1,2}(?:[-/]\d{1,2})?\b',flat)
result={'status':'nested_ADS_workbook_schema_preview_complete','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'raw_outer_ZIP_sha256':parent['raw_ZIP_sha256'],'nested_XLSX_sha256':nested_sha,'nested_XLSX_bytes':nested.stat().st_size,'all_sheet_names':workbook.sheetnames,'sheets_inspected':len(sheets),'sheet_previews':sheets,'header_date_tokens_first_10_rows':dates[:40],'vintage_date_roles_qualified':False,'new_HTTP_GETs':0,'market_outcome_reads':0,'Qlib_factor_rows':0,'model_fits':0,'native_accounts':0}
(card/'vintage_schema_audit.json').write_text(json.dumps(result,indent=2)+'\n')
(card/'schema_run_state.json').write_text(json.dumps({'status':'completed_bounded_schema_preview','plan_sha256':result['plan_sha256'],'outer_ZIP_reads':1,'nested_XLSX_extractions':1,'workbook_opens':1,'sheets_inspected':len(sheets),'model_fits':0,'accounts':0},indent=2)+'\n')
print({'sheets':workbook.sheetnames,'first_rows':[(s['name'],s['declared_max_row'],s['declared_max_column']) for s in sheets],'date_tokens':dates[:12]})

## 结论边界

预览不能代替完整版本日/观测日映射。版本定义和历史可用时钟须另冻限定审查。

本冻结 schema 预检未执行：同一 SHA 的原件已有独立 `schema_header_audit.json`，核查出 1,576 个唯一版本列和 2008–2026 日期覆盖。此卡不重复打开 1.47 GB 工作表 XML，见 `vintage_schema_duplicate_closeout.json`。